### Experiment 4
Develop an AI-powered system that automates resume screening using NLP to extract attributes like skills, experience, and qualifications. 
Apply rule-based filters for initial selection and use a neural network model trained on labeled resumes to classify candidates
into shortlisted, rejected or under review.


In [18]:
!pip install spacy

Defaulting to user installation because normal site-packages is not writeable


In [19]:
!python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [20]:
import spacy
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

In [21]:
# load the nlp pipeline
nlp = spacy.load('en_core_web_sm')

In [22]:
import random

roles = [
    "Software Engineer", "Junior Developer", "Senior Backend Engineer",
    "Data Scientist", "Receptionist", "Administrative Assistant",
    "Marketing Specialist", "IT Support", "Project Manager",
    "Graphic Designer", "Data Analyst", "Intern", "Web Developer",
    "Business Analyst", "Network Engineer", "UX Designer",
    "Content Writer", "Customer Service Representative", "QA Tester",
    "Systems Architect", "Mobile Developer", "DevOps Engineer",
    "Financial Analyst", "HR Specialist", "Operations Manager"
]

skills = [
    "Python", "Java", "C++", "JavaScript", "React", "HTML", "CSS",
    "SQL", "Excel", "WordPress", "Agile methodologies", "Adobe Creative Suite",
    "Machine Learning", "Embedded Systems", "Digital Campaigns", "APIs",
    "Troubleshooting", "Office Management", "Statistics", "Customer Service",
    "Cloud Computing", "Cybersecurity", "Data Visualization", "Project Planning"
]

degrees = [
    "Degree in CS", "Bachelor in Information Systems",
    "Master’s in Electrical Engineering", "Diploma in Web Development",
    "Bachelor’s in Statistics", "MBA in Marketing",
    "Associate Degree in IT", "PhD in Computer Science",
    "Certificate in Graphic Design", "Diploma in Business Administration",
    "Bachelor in Economics", "Master’s in Data Science",
    "Diploma in Networking", "Bachelor in Arts", "Certificate in UX Design"
]

def generate_resume():
    role = random.choice(roles)
    exp_years = random.randint(0, 15)
    skillset = random.sample(skills, 2)
    degree = random.choice(degrees)
    if exp_years == 0:
        return f"Intern with 6 months experience in {skillset[0]} and {skillset[1]}."
    else:
        return f"{role}, {exp_years} years experience in {skillset[0]} and {skillset[1]}. {degree}."

training_resumes = [generate_resume() for _ in range(500)]

# Preview first 20
for r in training_resumes[:20]:
    print(r)


Project Manager, 7 years experience in APIs and C++. Master’s in Data Science.
Systems Architect, 14 years experience in Data Visualization and React. Bachelor in Information Systems.
Project Manager, 8 years experience in CSS and Python. Bachelor in Economics.
Network Engineer, 12 years experience in HTML and Office Management. Bachelor in Arts.
Customer Service Representative, 12 years experience in Data Visualization and Cloud Computing. Certificate in UX Design.
Web Developer, 8 years experience in CSS and Machine Learning. Master’s in Data Science.
Operations Manager, 10 years experience in Adobe Creative Suite and Java. Degree in CS.
Intern, 5 years experience in APIs and Data Visualization. Diploma in Business Administration.
Data Scientist, 15 years experience in Digital Campaigns and SQL. Certificate in UX Design.
Customer Service Representative, 7 years experience in SQL and Cybersecurity. Associate Degree in IT.
IT Support, 5 years experience in Java and JavaScript. Diploma 

In [23]:
labels = [random.choice([0,1,2]) for _ in range(len(training_resumes))]

In [24]:
len(labels)

500

In [25]:
# nlp extraction - flattened into a list of "Features" text
processed_profiles = []
for text in training_resumes:
    doc = nlp(text)
    entities = [ent.text.lower() for ent in doc.ents if ent.label_ in ["ORG", "PRODUCT", "DATE", "GPE"]]
    # Combine original text with extracted focus keywords
    combined = text.lower() + " " + " ".join(entities)  # set() removes duplicates
    processed_profiles.append(combined)



In [26]:
processed_profiles

['project manager, 7 years experience in apis and c++. master’s in data science. 7 years data science',
 'systems architect, 14 years experience in data visualization and react. bachelor in information systems. 14 years data visualization and react',
 'project manager, 8 years experience in css and python. bachelor in economics. 8 years css economics',
 'network engineer, 12 years experience in html and office management. bachelor in arts. network engineer 12 years html office management arts',
 'customer service representative, 12 years experience in data visualization and cloud computing. certificate in ux design. customer service representative 12 years data visualization',
 'web developer, 8 years experience in css and machine learning. master’s in data science. 8 years css machine learning data science',
 'operations manager, 10 years experience in adobe creative suite and java. degree in cs. 10 years cs',
 'intern, 5 years experience in apis and data visualization. diploma in bus

In [27]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(processed_profiles, labels, test_size=0.35, random_state=42)

In [28]:
# Vectorization (coverting text to numbers)
vectorizer = CountVectorizer()
X_train= vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)    

In [29]:
# neural network model (multi-layer preceptron)
clf = MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42)

In [30]:
from sklearn.metrics import classification_report, accuracy_score

In [31]:
# Predictions
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Rejected", "Under Review", "Selected"]))


Accuracy: 0.2857142857142857
              precision    recall  f1-score   support

    Rejected       0.34      0.22      0.27        68
Under Review       0.33      0.26      0.29        61
    Selected       0.23      0.41      0.29        46

    accuracy                           0.29       175
   macro avg       0.30      0.30      0.29       175
weighted avg       0.31      0.29      0.28       175



In [32]:
new_resume = "4 years of Python experience and a Master's degree."
new_doc = nlp(new_resume)
new_entities = [ent.text.lower() for ent in new_doc.ents if ent.label_ in ["ORG", "PRODUCT", "DATE", "GPE"]]
new_features = vectorizer.transform([new_resume.lower() + " " + " ".join(new_entities)])

prediction = clf.predict(new_features)[0]
categories = {0 : "Rejected", 1 : "Under Review", 2 : "Selected"}

print(f"Candidate Resume : {new_resume}")
print(f"System Decision : {categories[prediction]}")


Candidate Resume : 4 years of Python experience and a Master's degree.
System Decision : Selected
